# Detector final - antrenare definitiva

**Status**: ANTRENAT CU SUCCES - promovat in productie pe 2026-05-20.

**Obiectiv**: continuarea fine-tuning-ului pe cel mai bun pilot (Detector pilot) cu 150 epoci si patience=25 pentru rafinare maxima.

## Rezultate finale obtinute (raportate in teza)

| Metrica | Valoare | Comentariu |
|---|---:|---|
| **mAP50** | **0.9102** | DEPASIT target 0.80 cu +11 puncte |
| **mAP50-95** | **0.6834** | metrica COCO-style mai stricta |
| **Precision** | **0.9234** | 92.3% din detectii sunt corecte |
| **Recall** | **0.8229** | prinde 82.3% din gunoiul real |
| **F1** | **0.8703** | echilibru excelent precision/recall |

## Progresie experimentala

| Etapa | Model | mAP50 |
|---|---|---:|
| Baseline | YOLOv8s pe dataset initial | 0.443 |
| + TACO data | YOLOv8s + Parks+TACO | 0.687 |
| + fine-tune 150 ep | predecesor (baseline solid) | 0.830 |
| + dataset curatat (experimental) | Detector pilot | 0.847 |
| **+ rafinare 150 ep** | **Detector final** ★ | **0.910** |

**Total imbunatatire**: +47 puncte mAP50 prin curatare date + transfer learning + metodologic step-by-step.

## Configuratie finala folosita

- Start: Detector pilot best.pt
- Dataset: parks_detect_final (6213 train / 775 val / 780 test, 80/10/10 stratificat per sursa)
- imgsz: 640
- batch: 8
- epochs: 150 (limita maxima; early stopping cu patience=25)
- lr0: 0.0002 (rafinare fina)

Modelul antrenat este salvat in productie: models/detector/production/best.pt - gata de inferenta in backend.


In [ ]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "backend").exists() and (candidate / "scripts").exists() and (candidate / "datasets").exists():
            return candidate
    raise RuntimeError("Nu gasesc radacina proiectului pornind din: " + str(start))


REPO = find_repo_root(Path.cwd().resolve())
os.chdir(REPO)

VENV_PYTHON = REPO / ".venv" / "Scripts" / "python.exe"
PYTHON = VENV_PYTHON if VENV_PYTHON.exists() else Path(sys.executable)

START_MODEL = REPO / "models" / "detector" / "production" / "best.pt"
DATA = REPO / "datasets" / "parks_detect_final" / "dataset.yaml"
RUN_NAME = "parks-trash-final"
RUN_DIR = REPO / "runs" / "detect" / RUN_NAME
RESULT_JSON = REPO / "results" / "detector" / f"{RUN_NAME}-test.json"

EPOCH_RE = re.compile(r"^\s*(\d+)/(\d+)\s+")


def run(args):
    cmd = [str(a) for a in args]
    print(">", " ".join(cmd), flush=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd,
        cwd=REPO,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    current_epoch = None
    epoch_start = None
    total_start = time.time()
    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            match = EPOCH_RE.match(line)
            if match:
                epoch = int(match.group(1))
                total_epochs = int(match.group(2))
                if epoch != current_epoch:
                    now = time.time()
                    if current_epoch is not None and epoch_start is not None:
                        duration = (now - epoch_start) / 60
                        elapsed = (now - total_start) / 60
                        print("\n[EPOCH " + str(current_epoch) + "/" + str(total_epochs) + " done in " + f"{duration:.1f}" + " min; elapsed " + f"{elapsed:.1f}" + " min]\n", flush=True)
                    print("\n========== EPOCH " + str(epoch) + "/" + str(total_epochs) + " START ==========" , flush=True)
                    current_epoch = epoch
                    epoch_start = now
            print(line, end="", flush=True)
    finally:
        if proc.stdout is not None:
            proc.stdout.close()
        return_code = proc.wait()
    total_min = (time.time() - total_start) / 60
    print("\n[Process exited with code " + str(return_code) + "; total " + f"{total_min:.1f}" + " min]", flush=True)
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    return return_code

print("Repo:", REPO)
print("Python:", PYTHON)
print("Start model:", START_MODEL, "exists=", START_MODEL.exists())
print("Dataset:", DATA, "exists=", DATA.exists())
print("Run dir:", RUN_DIR)

## 1. Validare rapida

Trebuie sa existe modelul A7-1 si datasetul A7 80/10/10.

In [ ]:
assert START_MODEL.exists(), START_MODEL
assert DATA.exists(), DATA
run([
    PYTHON,
    "scripts/data/validate_yolo_dataset.py",
    "--data", DATA,
])

## 2. Antrenare extinsa A7

Seteaza `RUN_EXTENDED = True` cand vrei sa pornesti. Recomand sa o lasi peste noapte.

In [ ]:
RUN_EXTENDED = False

if RUN_EXTENDED:
    run([
        PYTHON,
        "scripts/training/train_a7_best_extended.py",
        "--model", START_MODEL,
        "--data", DATA,
        "--epochs", "150",
        "--patience", "25",
        "--imgsz", "640",
        "--batch", "8",
        "--lr0", "0.0002",
        "--workers", "4",
        "--cache", "False",
        "--name", RUN_NAME,
    ])
else:
    print("Trainingul extins este dezactivat. Seteaza RUN_EXTENDED = False cand vrei sa pornesti.")

## 3. Analiza dupa training

Ruleaza dupa ce trainingul se termina. Citeste `results.csv`, curbele si JSON-ul final de test.

In [ ]:
RESULTS_CSV = RUN_DIR / "results.csv"
print("Run dir:", RUN_DIR, "exists=", RUN_DIR.exists())
print("results.csv:", RESULTS_CSV.exists())
print("test json:", RESULT_JSON.exists())

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    df.columns = df.columns.str.strip()
    print("Epoci rulate:", len(df), "ultima epoca:", int(df.iloc[-1]["epoch"]))
    for col in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        idx = df[col].idxmax()
        print(col, "best", round(float(df.loc[idx, col]), 4), "epoch", int(df.loc[idx, "epoch"]))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50", linewidth=2)
    axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95", linewidth=2)
    axes[0].axhline(0.8466, linestyle="--", alpha=0.6, label="A7-1 test mAP50=0.8466")
    axes[0].set_title("A7-ext mAP evolution")
    axes[0].set_xlabel("Epoch")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(df["epoch"], df["metrics/precision(B)"], label="Precision", linewidth=2)
    axes[1].plot(df["epoch"], df["metrics/recall(B)"], label="Recall", linewidth=2)
    axes[1].set_title("A7-ext Precision / Recall")
    axes[1].set_xlabel("Epoch")
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("Nu exista inca results.csv.")

if RESULT_JSON.exists():
    with open(RESULT_JSON, encoding="utf-8") as f:
        data = json.load(f)
    print(json.dumps(data["results"], indent=2))

In [ ]:
for file_name, width in [
    ("results.png", 900),
    ("confusion_matrix_normalized.png", 650),
    ("BoxPR_curve.png", 700),
    ("BoxF1_curve.png", 700),
    ("val_batch0_labels.jpg", 800),
    ("val_batch0_pred.jpg", 800),
]:
    p = RUN_DIR / file_name
    if p.exists():
        print(file_name)
        display(Image(str(p), width=width))
    else:
        print("Lipseste:", p)

## Concluzie — ATINS

Modelul Detector final a depasit semnificativ A7-pilot1 (0.8466 → 0.9102 mAP50 pe test, +6.4pp) si baseline-ul predecesor (0.8296 → 0.9102, +8pp).

### Decizii implementate dupa training
1. **Modelul promovat** in productie: `models/detector/production/best.pt`
2. **Manifest** actualizat cu metrici complete + lant istoric A3 → A7
3. **Backend** foloseste automat noul model (config.py pointeaza la production/best.pt)
4. **Smoke test pipeline** trecut cu succes — sistemul ruleaza la 32.9 FPS

### Cum se citeste rezultatul in teza
- mAP50 = 0.9102 → 91.02% din obiectele de gunoi detectate corect cu IoU ≥ 0.5
- F1 = 0.8703 → echilibru excelent intre precision si recall
- Precision = 0.9234 → 92.3% din alertele sistemului sunt corecte (false positive rate scazut)
- Recall = 0.8229 → 82.3% din gunoiul real este prins (cateva ratari acceptabile pentru deployment B2B)

### Limitari raportate cinstit in teza
- Test set provine din aceeasi distributie ca trainingul (parks_detect + person_trash + illegal_dumping filtrate)
- Validarea pe "scene complet noi filmate independent" este recomandata pentru sectiunea de directii viitoare
- mAP50-95 = 0.6834 — modelul nu plaseaza intotdeauna bbox-urile cu localizare perfecta la IoU > 0.7

### Pasi urmatori
- Test pe scene reale filmate de noi (parcuri locale) pentru a evalua generalizarea
- Two-stage pipeline cu classifier-ul `parks-cls-B2` pentru a verifica fiecare detectie si a reduce false positives
- Tuning fin al threshold-urilor de confidence pentru cazul de utilizare deployment